# Week 1 — PyTorch basics: solubility regression on ESOL

Goal: build the full training pipeline by hand — no Lightning, no wandb. See `README.md` for the full brief and hard constraints.

Pipeline: SMILES -> Morgan fingerprint -> `Dataset` -> `DataLoader` -> `nn.Module` -> training loop -> validation loop -> checkpoint.

Each section below has a markdown cell describing the step and a code cell with a stub. Fill in the `TODO`s yourself — don't skip to the answer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_absolute_error, r2_score

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

from datasets import load_dataset

# random seed and set it for numpy + torch, so your run is reproducible.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load the ESOL dataset

Search the HuggingFace Hub (https://huggingface.co/datasets) for an ESOL / MoleculeNet aqueous-solubility dataset. Don't assume a repo id — search for it and check the dataset card so you know what the columns actually are before you write code against them.

Once loaded, inspect it: how many rows, what are the column names, which column is the SMILES string, which column is the solubility target (and what units / scale is it on — raw or log solubility)?

In [89]:
# load dataset split into folds
ds_train = load_dataset("HR-machine/ESol", split="train").to_pandas()
ds_val = load_dataset("HR-machine/ESol", split="validation").to_pandas()
ds_test = load_dataset("HR-machine/ESol", split="test").to_pandas()
# inspect fold shapes, also look at one row to see column names and values.
print(f"train shape: {ds_train.shape}")
print(f"validation shape: {ds_val.shape}")
print(f"test shape: {ds_test.shape}")
print("="*10)
print("visualize dataset")
ds_train

train shape: (902, 10)
validation shape: (113, 10)
test shape: (113, 10)
visualize dataset


,smiles,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre
0,CC(C)=CCCC(C)=CC(=O),citral,-2.579,1,152.237,0,0,4,17.07,-2.06
1,CCCC=C,1-Pentene,-2.010,1,70.135,0,0,2,0.00,-2.68
2,CCCCCCCCCCCCCC,Tetradecane,-5.450,1,198.394,0,0,11,0.00,-7.96
3,CC(C)Cl,2-Chloropropane,-1.585,1,78.542,0,0,0,0.00,-1.41
4,CCC(C)CO,2-Methylbutanol,-1.027,1,88.150,1,0,2,20.23,-0.47
...,...,...,...,...,...,...,...,...,...,...
897,CC(=O)OCC(=O)C3(O)CCC4C2CCC1=CC(=O)CCC1(C)C2C(...,cortisone acetate,-3.426,1,402.487,1,4,3,97.74,-4.21
898,c3ccc2nc1ccccc1cc2c3,Acridine,-3.846,2,179.222,0,3,0,12.89,-3.67
899,Nc2cccc3nc1ccccc1cc23,1-aminoacridine,-3.542,1,194.237,1,3,0,38.91,-4.22
900,C1CCCCCC1,Cycloheptane,-2.916,2,98.189,0,1,0,0.00,-3.51


## 2. Featurize: SMILES -> Morgan fingerprint

Write a function that turns the SMILES strings into a fixed-length numeric vector using RDKit's Morgan fingerprint.

Then apply it across the whole dataset to build your feature matrix `X` and target vector `y`. Watch for SMILES strings that fail to parse.

In [104]:
fpgen = Chem.rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_morgan_fp(smiles) -> list:
    """Convert SMILES to a Morgan fingerprint bit vectors."""
    mols = Chem.MolFromSmiles(smiles)
    if mols is not None:
        return fpgen.GetFingerprintAsNumPy(mols)
    return None

ds_train['morgan_fp'] = ds_train['smiles'].apply(smiles_to_morgan_fp)
ds_val['morgan_fp'] = ds_val['smiles'].apply(smiles_to_morgan_fp)
ds_test['morgan_fp'] = ds_test['smiles'].apply(smiles_to_morgan_fp)

ds_train = ds_train.dropna(subset=['morgan_fp'])
ds_val = ds_val.dropna(subset=['morgan_fp'])
ds_test = ds_test.dropna(subset=['morgan_fp'])

print(len(ds_train),len(ds_val),len(ds_test))

902 113 113


## 3. Train / validation split

Split `X`/`y` into a training set and a validation set. Think about the split fraction and whether you need to set a random state for reproducibility.

- the ESol data set is already loaded from huggingface with fold splits so no need for this step

## 4. `Dataset` and `DataLoader`

Write a custom `torch.utils.data.Dataset` that wraps your feature/target arrays. It needs `__len__` and `__getitem__`. Think about dtypes: what dtype does `nn.Linear` expect, and what dtype are your fingerprint bits / targets in right now?

In [106]:
class ESOLDataset(Dataset):
    def __init__(self, dataset: pd.DataFrame):
        self.X = torch.from_numpy(np.stack(dataset['morgan_fp']))
        self.y = torch.from_numpy(np.stack(dataset['ESOL predicted log solubility in mols per litre']))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_i = self.X[idx]
        y_i = self.y[idx]
        return x_i, y_i

train_dataset = ESOLDataset(ds_train)
val_dataset = ESOLDataset(ds_val)
test_dataset = ESOLDataset(ds_train)

# wrap dataset in DataLoader. Pick a batch size. Should train and val/test use the same shuffle setting?
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, BATCH_SIZE, shuffle=False)

## 5. Model — `nn.Module`

A small MLP: input layer sized to your fingerprint length, two or three hidden layers, and a single output for the regression target. Think about what (if any) activation belongs on the output layer for a regression task, versus a classification one.

In [110]:
class SolubilityRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = None):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits

model = SolubilityRegressor(2048, 512)
print(model)

SolubilityRegressor(
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=2048, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1, bias=True)
  )
)


## 6. Training loop

Pick a loss function appropriate for regression, and an optimizer. Write the loop: for each epoch, iterate over `train_loader`, do the forward pass, compute loss, zero gradients, backward, step. Track the average training loss per epoch so you can plot it later.

In [ ]:
# TODO: pick a loss function for regression.
loss_fn = None

# TODO: pick an optimizer and learning rate.
optimizer = None

N_EPOCHS = None
train_losses = []

def train_one_epoch(model, loader, loss_fn, optimizer) -> float:
    """Run one training epoch, return the average training loss."""
    # TODO: model.train(), loop over batches, forward/backward/step, accumulate loss.
    raise NotImplementedError

# TODO: loop for N_EPOCHS, call train_one_epoch, append to train_losses, print progress.


## 7. Validation loop

Write a separate function that evaluates the model on `val_loader` without updating weights. What does the model need to be set to, and what context manager avoids wasting memory on gradients you won't use? Report validation loss plus at least one interpretable metric (RMSE, MAE, or R^2 - `sklearn.metrics` has these) alongside the raw loss.

In [ ]:
def evaluate(model, loader, loss_fn) -> dict:
    """Run validation, return a dict of metrics (loss, mae, r2, ...)."""
    # TODO: model.eval(), no_grad, loop over batches, collect predictions and targets.
    # TODO: compute loss and at least one of MAE / RMSE / R2 over the full val set.
    raise NotImplementedError


val_losses = []
# TODO: fold evaluate() into your training loop above (once per epoch), or re-run it here
# against the final model - your choice, but decide deliberately and note why in REFLECTION.md.

## 8. Checkpointing

Save the model's weights to disk (not the whole model object - just the `state_dict`, plus whatever else you'd need to resume: optimizer state, epoch number, best val loss). Then load it back into a fresh model instance and confirm it reproduces the same validation metric you got above.

In [ ]:
CHECKPOINT_PATH = "checkpoint.pt"

# TODO: save a checkpoint - what needs to be in the dict you save?

# TODO: load it back into a *new* model instance and re-run evaluate() to confirm the
# metrics match what you saw before saving.


## 9. Look at what you built

Plot train (and val, if you tracked it per-epoch) loss curves over epochs. Does it look like it's still improving, plateaued, or overfitting? Take this - and anything that broke along the way - into `REFLECTION.md`.

In [ ]:
# TODO: plot train_losses (and val_losses if you have them) vs epoch.
